In [6]:
import bz2
import re
import pandas as pd
from collections import defaultdict
import bz2

PATH = r"C:\Users\jaden\Downloads\sample.nt.bz2"

ENTITY_TO_STMT_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+)> '
    r'<http://www\.wikidata\.org/prop/(P\d+)> '
    r'<http://www\.wikidata\.org/entity/statement/([^>]+)> \.$'
)

STMT_TO_OBJ_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://www\.wikidata\.org/prop/statement/(P\d+)> '
    r'<http://www\.wikidata\.org/entity/(Q\d+)> \.$'
)

STMT_TO_TIME_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://www\.wikidata\.org/prop/qualifier/(P580|P582)> '
    r'"([^"]+)"\^\^<http://www\.w3\.org/2001/XMLSchema#dateTime> \.$'
)

STMT_DEPRECATED_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://wikiba\.se/ontology#rank> '
    r'<http://wikiba\.se/ontology#DeprecatedRank> \.$'
)

statements = defaultdict(dict)
line_count = 0

try:
    with bz2.open(PATH, "rt", encoding="utf-8") as f:
        for raw_line in f:
            line_count += 1
            line = raw_line.strip()

            m = ENTITY_TO_STMT_RE.match(line)
            if m:
                subj, pred, stmt_id = m.groups()
                statements[stmt_id]["subject"] = subj
                statements[stmt_id]["pred"] = pred
                continue

            m = STMT_TO_OBJ_RE.match(line)
            if m:
                stmt_id, pred2, obj = m.groups()
                statements[stmt_id]["obj"] = obj
                if "pred" in statements[stmt_id] and statements[stmt_id]["pred"] != pred2:
                    statements[stmt_id]["pred_mismatch"] = True
                else:
                    statements[stmt_id]["pred"] = pred2
                continue

            m = STMT_TO_TIME_RE.match(line)
            if m:
                stmt_id, time_prop, dt = m.groups()
                if time_prop == "P580":
                    statements[stmt_id]["start"] = dt
                else:
                    statements[stmt_id]["end"] = dt
                continue

            m = STMT_DEPRECATED_RE.match(line)
            if m:
                stmt_id = m.group(1)
                statements[stmt_id]["deprecated"] = True
                continue

except EOFError:
    print("Warning: truncated bz2 file reached. Using the valid portion that was read.")

rows = []
for stmt_id, data in statements.items():
    if data.get("deprecated", False):
        continue
    if data.get("pred_mismatch", False):
        continue
    if "subject" not in data or "pred" not in data or "obj" not in data:
        continue
    if "start" not in data and "end" not in data:
        continue

    rows.append({
        "subject": data["subject"],
        "pred": data["pred"],
        "obj": data["obj"],
        "start": data.get("start"),
        "end": data.get("end"),
    })

df = pd.DataFrame(rows, columns=["subject", "pred", "obj", "start", "end"])

print(f"Lines processed: {line_count:,}")
print(f"Temporal rows: {len(df):,}")
print(df.head())

Lines processed: 10,297,730
Temporal rows: 17,247
  subject  pred      obj                 start                   end
0     Q31   P38    Q4916  1999-01-01T00:00:00Z                  None
1     Q31   P38  Q232415  1830-01-01T00:00:00Z  2002-01-01T00:00:00Z
2     Q31  P530     Q212  1992-03-10T00:00:00Z                  None
3     Q31  P530     Q230  1992-06-05T00:00:00Z                  None
4     Q31  P530     Q423  2001-01-23T00:00:00Z                  None


In [ ]:
import pandas as pd

# If not already parsed, convert to datetime
df["start_dt"] = pd.to_datetime(df["start"], errors="coerce", utc=True)
df["end_dt"] = pd.to_datetime(df["end"], errors="coerce", utc=True)

has_start = df["start_dt"].notna()
has_end = df["end_dt"].notna()

only_start = (has_start & ~has_end).sum()
only_end = (~has_start & has_end).sum()
both = (has_start & has_end).sum()
neither = (~has_start & ~has_end).sum()

total = len(df)

print(f"Total rows: {total:,}")
print(f"Only start: {only_start:,}")
print(f"Only end:   {only_end:,}")
print(f"Both:       {both:,}")
print(f"Neither:    {neither:,}")

['Q4916', 'Q232415', 'Q212', 'Q230', 'Q423', 'Q1079522', 'Q12971', 'Q12967', 'Q55008046', 'Q12973', 'Q445553', 'Q12973', 'Q12976', 'Q12976', 'Q3911', 'Q155004', 'Q18434995', 'Q950958', 'Q476596', 'Q336599']
1.0
Total rows: 17,247
Only start: 6,830
Only end:   1,718
Both:       8,000
Neither:    699


In [16]:
# drop 
df = df[df["start"].notna() & df["end"].notna()].copy()
df.head()

,subject,pred,obj,start,end,start_dt,end_dt
1,Q31,P38,Q232415,1830-01-01T00:00:00Z,2002-01-01T00:00:00Z,1830-01-01 00:00:00+00:00,2002-01-01 00:00:00+00:00
5,Q31,P35,Q1079522,1831-02-25T00:00:00Z,1831-07-20T00:00:00Z,1831-02-25 00:00:00+00:00,1831-07-20 00:00:00+00:00
6,Q31,P35,Q12971,1831-06-04T00:00:00Z,1865-12-10T00:00:00Z,1831-06-04 00:00:00+00:00,1865-12-10 00:00:00+00:00
7,Q31,P35,Q12967,1865-12-17T00:00:00Z,1909-12-17T00:00:00Z,1865-12-17 00:00:00+00:00,1909-12-17 00:00:00+00:00
8,Q31,P35,Q55008046,1909-12-23T00:00:00Z,1934-02-17T00:00:00Z,1909-12-23 00:00:00+00:00,1934-02-17 00:00:00+00:00


In [ ]:
import bz2
import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd


DUMP_PATH = r"C:\Users\jaden\Downloads\sample.nt.bz2"
OUTPUT_JSON = r"C:\Users\jaden\Downloads\temporal_year_db.json"


LABEL_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+|P\d+)> '
    r'<http://www\.w3\.org/2000/01/rdf-schema#label> '
    r'"((?:[^"\\]|\\.)*)"@en \.$'
)

DESC_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+|P\d+)> '
    r'<http://schema\.org/description> '
    r'"((?:[^"\\]|\\.)*)"@en \.$'
)


def normalize_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith("http://www.wikidata.org/entity/"):
        s = s.rsplit("/", 1)[-1]
    return s


def year_from_value(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None

    i = 0
    if s[0] in "+-":
        i = 1

    digits = []
    while i < len(s) and s[i].isdigit():
        digits.append(s[i])
        i += 1

    if not digits:
        return None

    year = int("".join(digits))
    return -year if s[0] == "-" else year


def build_lookup_from_dump(dump_path, ids_needed):
    ids_needed = {normalize_id(x) for x in ids_needed if normalize_id(x) is not None}
    lookup = {eid: {"label": None, "description": None} for eid in ids_needed}
    complete = 0

    try:
        with bz2.open(dump_path, "rt", encoding="utf-8") as f:
            for i, line in enumerate(f, start=1):
                if "wikidata.org/entity/" not in line:
                    continue

                m = LABEL_RE.match(line)
                if m:
                    eid, label = m.groups()
                    if eid in lookup and lookup[eid]["label"] is None:
                        lookup[eid]["label"] = bytes(label, "utf-8").decode("unicode_escape")
                        if lookup[eid]["description"] is not None:
                            complete += 1
                    continue

                m = DESC_RE.match(line)
                if m:
                    eid, desc = m.groups()
                    if eid in lookup and lookup[eid]["description"] is None:
                        lookup[eid]["description"] = bytes(desc, "utf-8").decode("unicode_escape")
                        if lookup[eid]["label"] is not None:
                            complete += 1
                    continue

                if complete == len(ids_needed):
                    break

                if i % 5_000_000 == 0:
                    print(f"Processed {i:,} lines")

    except EOFError:
        print("Warning: dump appears truncated; returning partial lookup.")

    return lookup


def build_year_json(df, dump_path, output_json_path):
    required = {"subject", "pred", "obj", "start", "end"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df.copy()
    df["subject"] = df["subject"].map(normalize_id)
    df["pred"] = df["pred"].map(normalize_id)
    df["obj"] = df["obj"].map(normalize_id)

    df = df[df["start"].notna() & df["end"].notna()].copy()

    ids_needed = set(df["subject"]) | set(df["pred"]) | set(df["obj"])
    print(f"Need to resolve {len(ids_needed):,} unique subject/predicate IDs from dump...")

    lookup = build_lookup_from_dump(dump_path, ids_needed)

    def subject_info(qid):
        info = lookup.get(qid)
        if not info:
            return None
        label = info.get("label")
        description = info.get("description")
        if not label or not description:
            return None
        return {"label": label, "description": description}

    def relation_text(pid):
        info = lookup.get(pid)
        if not info:
            return None
        return info.get("label")


    df["head"] = df["subject"].map(subject_info)
    df["relation"] = df["pred"].map(relation_text)
    df["tail"] = df["obj"].map(subject_info)
    df["start_year"] = df["start"].map(year_from_value)
    df["end_year"] = df["end"].map(year_from_value)

    print("head:", df["head"].notna().sum())
    print("relation:", df["relation"].notna().sum())
    print("tail:", df["tail"].notna().sum())
    print("start_year:", df["start_year"].notna().sum())
    print("end_year:", df["end_year"].notna().sum())
    print("total rows:", len(df))

    print("head & relation:", (df["head"].notna() & df["relation"].notna()).sum())
    print("years ok:", (df["start_year"].notna() & df["end_year"].notna()).sum())

    df = df[
        df["head"].notna()
        & df["relation"].notna()
        & df["tail"].notna()
        & df["start_year"].notna()
        & df["end_year"].notna()
    ].copy()

    print(f"Rows with everything: {len(df)}")


    df["start_year"] = df["start_year"].astype(int)
    df["end_year"] = df["end_year"].astype(int)
    df = df[df["start_year"] <= df["end_year"]].copy()

    year_db = defaultdict(list)
    month_db = defaultdict(list)


    def parse_year_month(x):
        """
        Handles strings like:
        1843-##-##
        1843-05-##
        1843-05-01T00:00:00Z
        +1843-05-01T00:00:00Z

        Returns:
        (year, month) where month can be None if unknown
        """
        s = str(x).strip()

        if s[0] == "+":
            s = s[1:]

        year = int(s[:4])

        month = None
        if len(s) >= 7:
            mm = s[5:7]
            if mm.isdigit() and mm != "00":
                month = int(mm)

        return year, month


    for _, row in df.iterrows():
        triple = {
            "head": row["head"],
            "relation": row["relation"],
            "tail": row["tail"],
            "start": row["start"],
            "end": row["end"],
        }

        start_year, start_month = parse_year_month(row["start"])
        end_year, end_month = parse_year_month(row["end"])

        # Fill missing months for interval expansion
        start_month = 1 if start_month is None else start_month
        end_month = 12 if end_month is None else end_month

        # Year DB
        for year in range(start_year, end_year + 1):
            year_db[str(year)].append(triple)

        # Month DB
        y, m = start_year, start_month
        while (y < end_year) or (y == end_year and m <= end_month):
            month_db[f"{y:04d}-{m:02d}"].append(triple)

            m += 1
            if m == 13:
                m = 1
                y += 1

    year_db = dict(year_db)
    month_db = dict(month_db)

    output_json_path = Path(output_json_path)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(year_db, f, indent=2, ensure_ascii=False)
    
    output_json_path_monthly = output_json_path.with_name(
        output_json_path.stem + "_monthly.json"
    )
    with open(output_json_path_monthly, "w", encoding="utf-8") as f:
        json.dump(month_db, f, indent=2, ensure_ascii=False)

    print(f"Saved JSON DB to: {output_json_path}. _monthly for monthly one")
    print(f"Rows kept: {len(df):,}")
    print(f"Years written: {len(year_db):,}")
    print(f"Months written: {len(month_db):,}")
    print(f"Total triples: {sum(len(v) for v in year_db.values()):,}")
    

    return year_db, df, lookup


year_db, filtered_df, lookup = build_year_json(
    df=df,
    dump_path=DUMP_PATH,
    output_json_path=OUTPUT_JSON,
)

Need to resolve 7,511 unique subject/predicate IDs from dump...
Processed 10,000,000 lines
head: 8467
relation: 0
tail: 187
start_year: 8564
end_year: 8564
total rows: 8564
head & relation: 0
years ok: 8564
Saved JSON DB to: C:\Users\jaden\Downloads\temporal_year_db.json
Rows kept: 0
Years written: 0
Total year-indexed triples: 0


In [9]:
import json
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# User config
# =========================
DATA_PATH = r"C:\Users\jaden\Downloads\temporal_year_db.json"
YEAR = 1793
MAX_EDGES_TO_DRAW = 300
ONLY_LARGEST_COMPONENT = False


# =========================
# Helpers
# =========================
def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def maybe_reduce_for_plot(
    G: nx.MultiDiGraph,
    max_edges: int | None = None,
    largest_component_only: bool = False,
) -> nx.MultiDiGraph:
    H = G

    if largest_component_only and G.number_of_nodes() > 0:
        components = list(nx.connected_components(G.to_undirected()))
        if components:
            largest = max(components, key=len)
            H = G.subgraph(largest).copy()

    if max_edges is not None and H.number_of_edges() > max_edges:
        trimmed = nx.MultiDiGraph()
        for i, (u, v, k, data) in enumerate(H.edges(keys=True, data=True)):
            if i >= max_edges:
                break
            trimmed.add_node(u)
            trimmed.add_node(v)
            trimmed.add_edge(u, v, key=k, **data)
        H = trimmed

    return H


def build_dataframe(year_triples: list[dict]) -> pd.DataFrame:
    rows = []
    for t in year_triples:
        head = t.get("head", {})
        rows.append(
            {
                "subject_text": head.get("label"),
                "pred_text": t.get("relation"),
                "obj_text": t.get("tail"),
                "start": t.get("start"),
                "end": t.get("end"),
            }
        )

    df = pd.DataFrame(rows)
    df = df[
        df["subject_text"].notna()
        & df["pred_text"].notna()
        & df["obj_text"].notna()
    ].reset_index(drop=True)
    return df


def build_graph(df: pd.DataFrame) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()

    for _, row in df.iterrows():
        G.add_edge(
            row["subject_text"],
            row["obj_text"],
            label=row["pred_text"],
            start=row["start"],
            end=row["end"],
        )

    return G


def draw_graph(G: nx.MultiDiGraph, year: int) -> None:
    if G.number_of_nodes() == 0:
        print(f"No triples to display for {year}.")
        return

    plt.figure(figsize=(16, 12))
    pos = nx.spring_layout(G, seed=42, k=1.2)

    nx.draw_networkx_nodes(G, pos, node_size=900)
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edges(
        G,
        pos,
        arrows=True,
        arrowstyle="->",
        arrowsize=15,
        connectionstyle="arc3,rad=0.08",
    )

    edge_labels = {
        (u, v, k): data["label"]
        for u, v, k, data in G.edges(keys=True, data=True)
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8,
        rotate=False,
        label_pos=0.5,
    )

    plt.title(f"Wikidata temporal graph for {year}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# =========================
# Main
# =========================
def main():
    year_db = load_json(DATA_PATH)
    year_triples = year_db.get(str(YEAR), [])

    print(f"Triples for {YEAR}: {len(year_triples):,}")
    if not year_triples:
        return

    df = build_dataframe(year_triples)
    print("\nPreview:")
    print(df.head(20).to_string(index=False))

    G_full = build_graph(df)
    G_draw = maybe_reduce_for_plot(
        G_full,
        max_edges=MAX_EDGES_TO_DRAW,
        largest_component_only=ONLY_LARGEST_COMPONENT,
    )

    print(f"\nGraph nodes: {G_full.number_of_nodes():,}")
    print(f"Graph edges: {G_full.number_of_edges():,}")

    if G_draw.number_of_edges() != G_full.number_of_edges():
        print(
            f"Drawing reduced graph with {G_draw.number_of_nodes():,} nodes and "
            f"{G_draw.number_of_edges():,} edges."
        )

    draw_graph(G_draw, YEAR)
    return df, G_full


if __name__ == "__main__":
    df, G = main()

Triples for 1793: 0


TypeError: cannot unpack non-iterable NoneType object